# POMDPs and Q-learning — interactive companion

Companion to [Post 1d: POMDPs and Q-learning](../posts/01d-pomdps-and-q-learning.qmd).
Two themes in one notebook:
- **POMDPs**: when you can't observe the state directly. Belief updates,
  the Tiger problem.
- **Q-learning**: when you don't have a model at all. Learn the value
  function from samples.

**What you'll do (≈ 25 minutes):**
1. Watch belief updates in the Tiger problem.
2. Run Q-learning on a gridworld and compare to value iteration.
3. Discover that constant ε beats decaying ε on simple problems.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.pomdp import TigerPOMDP, belief_update
from nano_agents.mdp import (
    TwoGoalGridWorld, value_iteration, simulate_step,
    TabularQLearning, train_q_learning,
)
from nano_agents.mdp.visualization import plot_policy

## 1. The Tiger problem — Bayesian beliefs

Two doors. Behind one is a tiger; behind the other is gold. You can
**Listen** (-1, noisy hint about which side the tiger is) or **Open**
left/right (+10 if gold, -100 if tiger). The state is not observed —
your only handle is a probability distribution over "tiger left" vs
"tiger right".

Each listen, you update your belief by Bayes' rule.

In [ ]:
tiger = TigerPOMDP(listen_accuracy=0.85)
LISTEN = 2

# Track the belief over time as we observe a sequence of "growl from left" hints.
belief = np.array([0.5, 0.5])
history = [belief.copy()]
for _ in range(8):
    # Observation 0 = growl from left (suggesting tiger is on the left).
    belief = belief_update(belief, LISTEN, o=0, P=tiger.P, Z=tiger.Z)
    history.append(belief.copy())

history = np.array(history)
plt.plot(history[:, 0], "o-", label="P(tiger on left)")
plt.plot(history[:, 1], "s-", label="P(tiger on right)")
plt.axhline(0.5, color="black", linestyle="--", alpha=0.4, label="uniform prior")
plt.xlabel("listens (all hearing growl from LEFT)")
plt.ylabel("belief")
plt.title("Bayesian belief after consistent observations")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"After 8 consistent listens, P(tiger left) = {history[-1, 0]:.4f}")

The belief saturates at ~0.99996 — even 8 consistent listens at 85% accuracy
make us extremely confident. Open the right door!

### Try this
- Set `listen_accuracy=0.55` (barely informative). How many listens to
  reach 90% confidence?
- Mix observations: 6 lefts and 2 rights. Where does the belief end up?
- What if observations are *anti-correlated* (always growl from the
  opposite of where the tiger really is)? Run with `listen_accuracy=0.15`.

## 2. Q-learning on a gridworld

Now back to fully-observed MDPs, but without a model. Q-learning treats the
Bellman equation as a regression target: at each step, push $Q(s, a)$
toward $r + \gamma \max_{a'} Q(s', a')$.

In [ ]:
env = TwoGoalGridWorld(slip=0.1, step_reward=-0.04)
gamma = 0.95

# Reference: optimal value/policy from VI (gives us ground truth).
V_star, pi_star, _ = value_iteration(env, gamma=gamma)

# Train Q-learning.
agent = TabularQLearning(env.nS, env.nA, alpha=0.3, gamma=gamma, eps=0.1,
                          rng=np.random.default_rng(0))
hist = train_q_learning(env, agent, n_episodes=2000)
print(f"Final Q-value error vs V*: {np.max(np.abs(agent.Q.max(axis=1) - V_star)):.3f}")

# Compare policies.
pi_q = agent.Q.argmax(axis=1)
matches = sum(1 for s in env.states if not env.is_terminal(s)
               and pi_q[env.state_to_idx[s]] == pi_star[env.state_to_idx[s]])
n_nonterm = sum(1 for s in env.states if not env.is_terminal(s))
print(f"Policy matches VI on {matches}/{n_nonterm} non-terminal states.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_policy(env, pi_star, V=V_star, ax=axes[0])
axes[0].set_title("Value iteration (knows model)")
plot_policy(env, pi_q, V=agent.Q.max(axis=1), ax=axes[1])
axes[1].set_title(f"Q-learning (samples only) — {matches}/{n_nonterm} match")
plt.tight_layout(); plt.show()

Q-learning recovers the optimal policy from samples alone — no transition
or reward model needed. The disagreements (if any) are typically at states
where multiple actions are similarly good.

### Try this
- Set `alpha=0.01`. Slower but reaches lower asymptotic error.
- Set `alpha=0.9`. Fast but stays noisy — Q-values oscillate forever.
- Set `eps=0.5`. With aggressive exploration, does Q-learning find a
  better or worse policy?

## 3. The exploration schedule trap

Textbooks recommend *decaying* $\varepsilon$ from 1.0 to 0.05 — explore
broadly early, exploit later. Sometimes a constant schedule works better.
Let's race them.

In [ ]:
n_episodes = 2000
n_seeds = 8

def train_with_schedule(eps_at):
    returns_avg = np.zeros(n_episodes)
    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        agent = TabularQLearning(env.nS, env.nA, alpha=0.3, gamma=gamma,
                                  eps=1.0, rng=rng)
        non_terminal = [s for s in env.states if not env.is_terminal(s)]
        returns = np.zeros(n_episodes)
        for ep in range(n_episodes):
            agent.eps = eps_at(ep)
            s = non_terminal[rng.integers(len(non_terminal))]
            ep_R = 0.0
            for _ in range(200):
                si = env.state_to_idx[s]
                a = agent.select(si)
                s_next, r, done = simulate_step(env, s, a, rng)
                si_next = env.state_to_idx[s_next]
                agent.update(si, a, r, si_next, done)
                ep_R += r
                s = s_next
                if done: break
            returns[ep] = ep_R
        returns_avg += returns / n_seeds
    return returns_avg

schedules = {
    r"constant $\varepsilon = 0.1$":          lambda ep: 0.1,
    r"linear decay 1.0 → 0.05":              lambda ep: max(0.05, 1.0 - ep / 1000),
    r"exponential decay (rate 0.001)":       lambda ep: max(0.05, np.exp(-ep * 0.001)),
}
results = {name: train_with_schedule(fn) for name, fn in schedules.items()}

w = 50
for name, ret in results.items():
    ma = np.convolve(ret, np.ones(w)/w, mode="valid")
    plt.plot(ma, label=name)
plt.xlabel("episode"); plt.ylabel(f"return ({w}-ep MA)")
plt.title("Q-learning under three exploration schedules")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Honest result** (also in Post 1d §8): constant $\varepsilon = 0.1$ wins
decisively on small problems. The decaying schedules pay a heavy cost
early on by being *too random* — they sample bad actions far more than
necessary.

The lesson generalizes: **aggressive exploration is for hard problems with
deceptive local minima**. On easy problems, conservative exploration plus
plenty of episodes is a better strategy.

### Try this
- Use a much harder problem: large grid + sparse reward + slippery
  transitions. Does the verdict reverse?
- Try a *Boltzmann* (softmax) policy on Q-values instead of ε-greedy.
  Temperature controls exploration smoothly.
- What happens if you set the *initial* ε very low, say 0.01? Does
  Q-learning still converge?

## What's next

Topic 1 is done. You've built:
- Bandits and contextual bandits (one-step decision problems).
- MDPs solved with value/policy iteration (model-based).
- POMDPs (partial observability).
- Q-learning (model-free).

Topic 2 generalizes to **policy gradient** methods, which don't need to
estimate Q at all — they directly optimize the policy via stochastic
gradient ascent on expected return. Open
[`02a-reinforce.ipynb`](02a-reinforce.ipynb) for the foundational
algorithm.